In [2]:
from sctoolbox.utils.jupyter import bgcolor, _compare_version

# change the background of input cells
bgcolor("PowderBlue", select=[2, 4])

nb_name = "0A_tobias.ipynb"

_compare_version(nb_name)

#  05 - TOBIAS
<hr style="border:2px solid black"> </hr>

## 1 - Description

This notebook provides a streamlined interface for ATAC-seq footprinting analysis using the [TOBIAS](https://github.com/loosolab/TOBIAS) toolkit, wrapped via custom functions that leverage [TOBIAS-Snakemake](https://github.com/loosolab/TOBIAS_snakemake).

 **Key features:**

 * **Tn5 bias correction** — Adjusts for insertion bias in ATAC-seq libraries.
 * **Footprint scoring** — Computes footprint strength across regulatory regions.
 * **Binding site inference** — Distinguishes between bound and unbound transcription factor (TF) sites.
 * **Visualization** — Generates footprint plots for individual conditions or comparative analyses.

 At the end of this notebook, a TOBIAS configuration YAML file is generated.
 Since the full pipeline is compute-intensive, please execute it from the command line as described in the **Running the Pipeline** section below.

 For complete documentation, refer to:

 * [TOBIAS repository](https://github.com/loosolab/TOBIAS)
 * [TOBIAS Wiki](https://github.com/loosolab/TOBIAS/wiki/) for more information
 * [TOBIAS-Snakemake repository](https://github.com/loosolab/TOBIAS_snakemake)


---

## 2 - Setup

In [3]:
import pandas as pd
pd.set_option('display.max_columns', None)  # no limit to the number of columns shown
import sctoolbox
from sctoolbox import settings
import sctoolbox.utils as utils
import sctoolbox.tools as tools
import os

sctoolbox.settings.settings_from_config("config.yaml", key="0A")

[WARNING] Log file '../logs/tobias_log.txt' already exists. The file will be overwritten since 'overwrite_log' is set to True.


---

## 3 - Load anndata

<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [4]:
# Input/Output
last_notebook_adata = "anndata_annotated.h5ad"

---

In [5]:
adata = utils.adata.load_h5ad(last_notebook_adata)

with pd.option_context("display.max.rows", 5, "display.max.columns", None):
    display(adata)
    display(adata.obs)

/mnt/workspace2/hschult/.conda/sctoolbox/lib/python3.12/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


[INFO] The adata object was loaded from: ../adatas/anndata_annotated.h5ad


AnnData object with n_obs × n_vars = 8803 × 160363
    obs: 'sample', 'disease state', 'tissue', 'selected cell types', 'fld_score', 'mean_fragment_size', 'n_fragments', 'fold_change_promoters_fragments', 'frip', 'tsse_score', 'doublet_score', 'predicted_doublet', 'n_features', 'log1p_n_features', 'leiden', 'LISI_score_X_pca', 'LISI_score_X_umap', 'sample_density', 'leiden_0.1', 'leiden_0.2', 'leiden_0.3', 'leiden_0.4', 'leiden_0.5', 'leiden_0.6', 'leiden_0.7', 'leiden_0.8', 'leiden_0.9', '0.2_1', '0.2_2', 'clustering', 'MR_symbol_clustering', 'SCSA_symbol_clustering', 'MR_score_clustering', 'SCSA_score_clustering', 'MR_ui_clustering', 'SCSA_ui_clustering'
    var: 'chr', 'start', 'end', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'annotation_feature', 'gene_strand', 'gene_start', 'gene_end', 'annotation_query', 'query_name', 'distance_to_gene', 'gene_anchor', 'gene_ovl_peak', 'peak_ovl_gene', 'relative_locatio

,sample,disease state,tissue,selected cell types,fld_score,mean_fragment_size,n_fragments,fold_change_promoters_fragments,frip,tsse_score,doublet_score,predicted_doublet,n_features,log1p_n_features,leiden,LISI_score_X_pca,LISI_score_X_umap,sample_density,leiden_0.1,leiden_0.2,leiden_0.3,leiden_0.4,leiden_0.5,leiden_0.6,leiden_0.7,leiden_0.8,leiden_0.9,0.2_1,0.2_2,clustering,MR_symbol_clustering,SCSA_symbol_clustering,MR_score_clustering,SCSA_score_clustering,MR_ui_clustering,SCSA_ui_clustering
barcode,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
AAACGAAAGAGAGGTA-1,sample 1,healthy,blood,PBMCs,1236.596857,131.17,55681,0.385877,0.755841,7.305137,0.048303,False,12605,9.441928,0,1.0,1.0,0.452229,1,1,1,1,1,1,1,1,1,1,1,1,CD1C-CD141- dendritic cell,CD1C-CD141- dendritic cell,CD1C-CD141- dendritic cell,CD1C-CD141- dendritic cell,CD1C-CD141- dendritic cell,CD1C-CD141- dendritic cell
AAACGAAAGCAGGAGG-1,sample 1,healthy,blood,PBMCs,1411.451173,123.53,18024,0.557725,0.764383,7.814169,0.073210,False,5392,8.592857,1,1.0,1.0,0.594502,2,2,3,4,3,3,3,4,3,5,7,7,Central memory CD4+ T cell,Regulatory T (Treg) cell,Multilymphoid progenitor cell,Regulatory T (Treg) cell,Multilymphoid progenitor cell,Regulatory T (Treg) cell
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTTGTGTTCTGATCTT-1,sample 1,healthy,blood,PBMCs,1669.071202,135.55,49822,0.329239,0.692217,4.656945,0.130035,False,13992,9.546312,0,1.0,1.0,0.699981,1,10,2,3,14,2,2,3,15,1,1,1,CD1C-CD141- dendritic cell,CD1C-CD141- dendritic cell,CD1C-CD141- dendritic cell,CD1C-CD141- dendritic cell,CD1C-CD141- dendritic cell,CD1C-CD141- dendritic cell
TTTGTGTTCTTACTCA-1,sample 1,healthy,blood,PBMCs,2070.700868,144.87,51305,0.381827,0.599754,6.906920,0.341463,False,10882,9.294957,6,1.0,1.0,0.159993,7,11,10,15,15,15,17,18,16,2,4,4,Activated CD8+ T cell,Natural killer cell,AXL+SIGLEC6+ dendritic cell,Natural killer cell,AXL+SIGLEC6+ dendritic cell,Natural killer cell


---

## 4 - General input

Set the parameters to build the TOBIAS config YAML file.

<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [6]:
# Set the column to evaluate. Pseudobulk bamfiles are created based on this column. 
groupby = "SCSA_ui_clustering"

# Barcode column  of the adata.obs table. If None index is used.
barcode_column = None

# Ouput directory
output = "../tobias/"

# The path to the ATAC experiment BAM file
# Must contain all cells
path_bam = "../../../data/ATAC/10k_pbmc_ATACv2_nextgem_Chromium_Controller_possorted_bam.bam"

# The path to the FASTA file for the organism
fasta = "../../../data/ATAC/homo_sapiens.110.mainChr.fa"

# The path to the (uncompressed) GTF file for the organism
gtf = "../../../data/ATAC/homo_sapiens.110.genes.gtf"

# The path to the blacklist file to use in the TOBIAS run (optional)
# If 'None', a mock blacklist file will be generated for the run
blacklist = '../../../data/ATAC/hg38.blacklist.v2.bed'

# The Path to the motif file for the organism (JASPAR or MEME)
motifs = "../../../data/ATAC/JASPAR2024_CORE_vertebrates_non-redundant_pfms_jaspar.txt"

# If the ATAC modality already has a column with read tags matching the tags used in the bam file
# give the name of the column here
# Must match "ATAC:<name of coulmn in anndata.obs table>"
bam_barcodes = 'CB'

# If bam_barcodes is None give the name of the column that contains the raw ATAC barcodes
# Must match "ATAC:<name of column in anndata.obs table>"
raw_barcodes_ATAC = None

# Name of the organism from which the data stems
# options = ["mouse", "human", "zebrafish"]
organism = "human"

# Give the name of the TOBIAS config yaml file in the format of "<name of file>.yml"
# It cannot be 'config.yml' 
yaml = "TOBIAS_config.yml"

---

## 5 - Prepare TOBIAS run

In [7]:
input_dir, _, yml = tools.tobias.prepare_tobias(adata,
                                                groupby,
                                                output=output,
                                                path_bam=path_bam,
                                                barcode_column=barcode_column,
                                                barcode_tag='CB',
                                                fasta=fasta,
                                                motifs=motifs,
                                                gtf=gtf,
                                                blacklist=blacklist,
                                                organism=organism,
                                                yml=yaml,
                                                plot_comparison=True,
                                                plot_correction=True,
                                                plot_venn=True,
                                                coverage=False,
                                                wilson=False,
                                                threads=4)

config_yaml = os.path.join(input_dir, yml)

[INFO] calculating barcode overlap between bamfile and adata.obs
[WARNING] 39 (3.90%) of the first 1000 reads did not have a 'CB' tag.
[INFO] Barcode tag: OK
[INFO] Found 5 groups in .obs.SCSA_ui_clustering: ['Naive CD8+ T cell', 'Natural killer cell', 'Regulatory T (Treg) cell', 'B cells', 'CD1C-CD141- dendritic cell']
[INFO] Reading total number of reads from bams...
[INFO] Starting splitting of bams...
 

Reading from bams:   0%|          | 0/1132667476 [00:00<?, ?reads/s]

Writing queued reads:   0%|          | 0/1 [00:00<?, ?reads/s]

IOPub message rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_msg_rate_limit`.

Current values:
NotebookApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
NotebookApp.rate_limit_window=3.0 (secs)

IOPub message rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_msg_rate_limit`.

Current values:
NotebookApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
NotebookApp.rate_limit_window=3.0 (secs)

IOPub message rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_msg_rate_limit`.

Current values:
NotebookApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
NotebookApp.rate_limit_window=3.0 (secs)

IOPub message rate exceed

[INFO] Finished splitting bams!
Writing TOBIAS config yaml.
[WARNING] Condition name Naive_CD8+_T_cell contains characters not in '[a-zA-Z0-9-_.]'. Replacing with '_'.
Wrote TOBIAS config yaml to '/mnt/workspace2/hschult/sc-framework-paper/Code_Schultheis_et_al_2025_SC-Framework/ATAC/analysis/01-analysis-run/tobias/input/TOBIAS_config.yml'


---

## 6. Executing the TOBIAS Pipeline

This section outlines how to launch the TOBIAS-Snakemake workflow using the configuration file generated above.

More information on TOBIAS-Snakemake can be found in the [TOBIAS-Snakemake Wiki](https://github.com/loosolab/TOBIAS_snakemake/wiki).

 **Prerequisites:**

 * Clone the [TOBIAS-Snakemake](https://github.com/loosolab/TOBIAS_snakemake) repository.
 * Ensure [TOBIAS-Snakemake](https://github.com/loosolab/TOBIAS_snakemake) is installed (see “Getting Started” in the repository’s [README](https://github.com/loosolab/TOBIAS_snakemake)).
 * Verify that Conda is available for creating rule-specific environments.

 **Notes:**

 * The `--use-conda` flag automates environment setup for each rule.
 * Tweak `--cores` based on your available CPU resources.
 * For advanced options or troubleshooting, refer to the TOBIAS-Snakemake documentation.

 **Command:**
 In your terminal, move to the [TOBIAS-Snakemake](https://github.com/loosolab/TOBIAS_snakemake) directory and execute the following command:

In [9]:
print('Bash Command:')
print(f"""snakemake \\
   --configfile {config_yaml} \\
   --use-conda \\
   --cores 10""")

Bash Command:
snakemake \
   --configfile /mnt/workspace2/hschult/sc-framework-paper/Code_Schultheis_et_al_2025_SC-Framework/ATAC/analysis/01-analysis-run/tobias/input/TOBIAS_config.yml \
   --use-conda \
   --cores 10
